# 🏥 Insurance Customer Churn Prediction
**Author:** Yuxuan Li  
**Tools:** Python, pandas, scikit-learn, matplotlib, seaborn  

---

## 🎯 Objective
Predict which insurance customers are likely to churn (cancel their policy), and identify the key drivers of churn to support retention strategies.

---

## 0. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

print('Libraries loaded successfully!')

## 1. Load & Explore Data
We use a dataset of 2,000 insurance customers with features covering demographics, policy details, claims history, and customer behaviour.

In [ ]:
df = pd.read_csv('insurance_churn.csv')

print(f'Dataset shape: {df.shape}')
print(f'Churn rate: {df["churn"].mean():.1%}')
df.head()

In [ ]:
# Check data types and missing values
print('Data types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())

In [ ]:
# Summary statistics
df.describe()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Insurance Churn – Exploratory Data Analysis', fontsize=16, fontweight='bold')

palette = {0: '#2E6DA4', 1: '#E05C2A'}
labels  = {0: 'Retained', 1: 'Churned'}

# 1. Churn distribution
ax = axes[0, 0]
counts = df['churn'].value_counts()
bars = ax.bar(['Retained', 'Churned'], counts.values, color=['#2E6DA4', '#E05C2A'], edgecolor='white')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10, str(val), ha='center', fontweight='bold')
ax.set_title('Churn Distribution', fontweight='bold')
ax.set_ylabel('Count')

# 2. Tenure vs Churn
ax = axes[0, 1]
for churn_val, group in df.groupby('churn'):
    ax.hist(group['tenure_years'], bins=15, alpha=0.6, label=labels[churn_val], color=palette[churn_val])
ax.set_title('Tenure Years by Churn', fontweight='bold')
ax.set_xlabel('Tenure (years)')
ax.legend()

# 3. Complaint count vs Churn rate
ax = axes[0, 2]
churn_by_complaint = df.groupby('complaint_count')['churn'].mean() * 100
ax.bar(churn_by_complaint.index, churn_by_complaint.values, color='#E05C2A', edgecolor='white')
ax.set_title('Churn Rate by Complaint Count', fontweight='bold')
ax.set_xlabel('Number of Complaints')
ax.set_ylabel('Churn Rate (%)')

# 4. Policy type vs Churn rate
ax = axes[1, 0]
churn_by_policy = df.groupby('policy_type')['churn'].mean() * 100
ax.bar(churn_by_policy.index, churn_by_policy.values, color=['#2E6DA4','#4A9DBF','#E05C2A','#F0A060'], edgecolor='white')
ax.set_title('Churn Rate by Policy Type', fontweight='bold')
ax.set_xlabel('Policy Type')
ax.set_ylabel('Churn Rate (%)')

# 5. Premium increase vs Churn
ax = axes[1, 1]
for churn_val, group in df.groupby('churn'):
    ax.hist(group['premium_increase_pct'], bins=15, alpha=0.6, label=labels[churn_val], color=palette[churn_val])
ax.set_title('Premium Increase % by Churn', fontweight='bold')
ax.set_xlabel('Premium Increase (%)')
ax.legend()

# 6. Correlation heatmap
ax = axes[1, 2]
numeric_cols = df.select_dtypes(include=np.number).columns
corr = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, ax=ax, cmap='coolwarm', center=0,
            fmt='.2f', annot=True, annot_kws={'size': 7}, linewidths=0.5)
ax.set_title('Feature Correlation Heatmap', fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Feature Engineering
We create three new features that capture business-relevant patterns not directly visible in the raw data.

In [ ]:
df_model = df.copy()

# Encode categorical variables as numbers
le = LabelEncoder()
df_model['policy_type_enc'] = le.fit_transform(df_model['policy_type'])
df_model['region_enc']      = le.fit_transform(df_model['region'])

# Create new features
df_model['claim_frequency']   = df_model['num_claims'] / (df_model['tenure_years'] + 1)
df_model['premium_per_policy']= df_model['annual_premium'] / df_model['num_policies']
df_model['high_risk']         = ((df_model['complaint_count'] > 0) & (df_model['num_claims'] > 1)).astype(int)

print('New features:')
print('  claim_frequency    = claims per year of tenure')
print('  premium_per_policy = premium / number of policies')
print('  high_risk          = 1 if customer has complaints AND multiple claims')
print(f'\nHigh-risk customers: {df_model["high_risk"].sum()} ({df_model["high_risk"].mean():.1%})')

## 4. Train/Test Split & Scaling

In [ ]:
feature_cols = [
    'age', 'tenure_years', 'num_policies', 'annual_premium',
    'num_claims', 'claim_amount', 'complaint_count', 'contacted_support',
    'premium_increase_pct', 'policy_type_enc', 'region_enc',
    'claim_frequency', 'premium_per_policy', 'high_risk'
]

X = df_model[feature_cols]
y = df_model['churn']

# 80% train, 20% test — stratified to preserve churn ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features (important for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Training set : {X_train.shape[0]} samples')
print(f'Test set     : {X_test.shape[0]} samples')

## 5. Train & Evaluate Models
We train three models and compare them using **ROC-AUC** — the best metric for imbalanced classification problems like churn.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting'  : GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = {}
for name, model in models.items():
    X_tr = X_train_scaled if name == 'Logistic Regression' else X_train
    X_te = X_test_scaled  if name == 'Logistic Regression' else X_test

    model.fit(X_tr, y_train)
    y_pred  = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    cv  = cross_val_score(model,
                          X_train_scaled if name == 'Logistic Regression' else X_train,
                          y_train, cv=5, scoring='roc_auc').mean()

    results[name] = {'model': model, 'y_pred': y_pred, 'y_proba': y_proba,
                     'accuracy': acc, 'auc': auc, 'cv_auc': cv}
    print(f'{name:25s}  Accuracy: {acc:.3f}  ROC-AUC: {auc:.3f}  CV-AUC: {cv:.3f}')

In [ ]:
# ROC curves + confusion matrix
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Model Performance Comparison', fontsize=15, fontweight='bold')

colors_roc = ['#2E6DA4', '#E05C2A', '#2EAA6D']
ax = axes[0]
for (name, res), col in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    ax.plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})", color=col, lw=2)
ax.plot([0,1],[0,1],'k--', lw=1, label='Random')
ax.set_title('ROC Curves', fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

names  = list(results.keys())
aucs   = [results[n]['auc']    for n in names]
cv_aucs= [results[n]['cv_auc'] for n in names]
x = np.arange(len(names))
w = 0.35
ax = axes[1]
bars1 = ax.bar(x - w/2, aucs,    w, label='Test AUC', color='#2E6DA4', edgecolor='white')
bars2 = ax.bar(x + w/2, cv_aucs, w, label='CV AUC',   color='#E05C2A', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels([n.replace(' ', '\n') for n in names], fontsize=9)
ax.set_ylim(0.5, 1.0)
ax.set_title('Test AUC vs CV AUC', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

best_name = max(results, key=lambda n: results[n]['auc'])
cm = confusion_matrix(y_test, results[best_name]['y_pred'])
ConfusionMatrixDisplay(cm, display_labels=['Retained','Churned']).plot(ax=axes[2], colorbar=False, cmap='Blues')
axes[2].set_title(f'Confusion Matrix\n{best_name}', fontweight='bold')

plt.tight_layout()
plt.show()
print(f'\n✅ Best model: {best_name} (AUC = {results[best_name]["auc"]:.3f})')

## 6. Feature Importance

In [ ]:
rf_model = results['Random Forest']['model']
importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors_fi = ['#E05C2A' if imp > importances.quantile(0.75) else '#2E6DA4' for imp in importances.values]
ax.barh(importances.index, importances.values, color=colors_fi, edgecolor='white')
ax.set_title('Feature Importance – Random Forest\n(Orange = Top 25% most important)', fontweight='bold', fontsize=13)
ax.set_xlabel('Importance Score')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print('\nTop 5 churn drivers:')
for i, (feat, val) in enumerate(importances.tail(5).iloc[::-1].items(), 1):
    print(f'  {i}. {feat}: {val:.3f}')

## 7. Business Risk Segmentation

In [ ]:
df_model['churn_prob']    = results['Gradient Boosting']['model'].predict_proba(df_model[feature_cols])[:, 1]
df_model['risk_segment']  = pd.cut(df_model['churn_prob'],
                                    bins=[0, 0.1, 0.25, 0.5, 1.0],
                                    labels=['Low Risk', 'Medium Risk', 'High Risk', 'Critical'])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Business Insights: Customer Risk Segmentation', fontsize=14, fontweight='bold')

seg_colors = ['#2EAA6D', '#F0C040', '#E05C2A', '#8B0000']
seg_counts  = df_model['risk_segment'].value_counts()

axes[0].pie(seg_counts.values, labels=seg_counts.index, autopct='%1.1f%%',
            colors=seg_colors, startangle=90, pctdistance=0.75)
axes[0].set_title('Customer Risk Segments', fontweight='bold')

seg_order = ['Low Risk', 'Medium Risk', 'High Risk', 'Critical']
premium_at_risk = df_model.groupby('risk_segment')['annual_premium'].sum() / 1_000
vals = [premium_at_risk.get(s, 0) for s in seg_order]
bars = axes[1].bar(seg_order, vals, color=seg_colors, edgecolor='white')
for bar, val in zip(bars, vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'€{val:.0f}K', ha='center', fontweight='bold', fontsize=9)
axes[1].set_title('Total Annual Premium at Risk by Segment', fontweight='bold')
axes[1].set_ylabel('Total Premium (thousands €)')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Conclusions & Business Recommendations

### Model Performance
- **Best model:** Logistic Regression (ROC-AUC = 0.765)
- All models outperform random prediction significantly
- Cross-validation confirms results are stable and not overfitted

### Key Churn Drivers
1. **Premium increase %** — customers are most sensitive to price hikes at renewal
2. **Annual premium** — higher-value customers churn more (possibly price-shopping)
3. **Claim frequency** — customers who claim often may feel underserved
4. **Tenure** — newer customers are at higher risk
5. **Complaints** — even one complaint dramatically raises churn probability

### Recommended Actions
| Segment | Action |
|---|---|
| 🔴 Critical | Immediate personal outreach, loyalty offer |
| 🟠 High Risk | Proactive renewal call, review premium increase |
| 🟡 Medium Risk | Automated engagement, satisfaction survey |
| 🟢 Low Risk | Standard renewal process |

> **Bottom line:** Resolving complaints quickly and being transparent about premium changes are the two most impactful levers for reducing churn.